# Final CatBoost Training

**ISIC 2024 – Skin Cancer Detection with 3D-TBP**

This notebook trains the final CatBoost model using the complete ISIC 2024 metadata dataset.

Unlike the cross-validation notebook, no validation split is created. The model architecture and hyperparameters have already been selected through five-fold cross-validation, so the objective is to obtain the final metadata model used during inference.

## Why train a final model?

Cross-validation is used to estimate the model performance and validate the selected hyperparameters. After the training configuration has been finalized, a new CatBoost model is trained using the complete metadata dataset.

The resulting model is the one used to generate metadata predictions for the competition test set.

## Configuration

Define dataset paths, cross-validation settings, and CatBoost hyperparameters used throughout the notebook.

In [1]:
# ==========================================================
# Paths
# ==========================================================

DATA_PATH = "../data/train-metadata.csv"

MODEL_DIR = "../models"

FEATURES_PATH = f"{MODEL_DIR}/features.json"

# ==========================================================
# CatBoost Hyperparameters
# ==========================================================

SEED = 42
ITERATIONS = 1000
LEARNING_RATE = 0.05
DEPTH = 7
EARLY_STOPPING = 100
AUTO_CLASS_WEIGHTS = "Balanced"
EVAL_METRIC = "AUC"
VERBOSE = 100

## Imports

Import the libraries and project modules required for preprocessing, training, evaluation, and saving outputs.

In [5]:
import numpy as np
import pandas as pd
import sys

from catboost import CatBoostClassifier
from pathlib import Path

sys.path.append(
    "../src/cat/"
)

from cat_dataset import load_data, prepare_data
from cat_metrics import evaluate
from cat_utils import (
    save_features,
    save_model,
    save_oof,
)
from cat_inference import predict_all_folds
from cat_utils import predict_model

## Dataset Preparation

Load the ISIC 2024 metadata, perform preprocessing and feature engineering, and identify categorical features for CatBoost.

In [3]:
print("Loading dataset...")

df = load_data(DATA_PATH)

X, y, groups, cat_features = prepare_data(df)

Loading dataset...
Loading dataset...


In [6]:
pred_cv = predict_all_folds(X=X, model_dir="../5fold-cv-models",n_folds=5)
pred_final = predict_model(X=X, model_path="../models/catboost_final.cbm")

print(evaluate(y, pred_cv))

print(evaluate(y, pred_final))

Predicting fold 0...
Predicting fold 1...
Predicting fold 2...
Predicting fold 3...
Predicting fold 4...
{'roc_auc': 0.9912524971621994, 'pauc': 0.1937595874878505}
{'roc_auc': 1.0, 'pauc': 0.19999999999999996}


## Save Feature Information

Store the feature names and categorical feature list so that the inference pipeline can reproduce the same preprocessing later.

In [5]:
save_features(
    features=X.columns.tolist(),
    categorical_features=cat_features,
    path=FEATURES_PATH,
)

## Model Initialization

Instantiate the final CatBoost classifier using the selected hyperparameters.

In [6]:
model = CatBoostClassifier(
    iterations=ITERATIONS,
    learning_rate=LEARNING_RATE,
    depth=DEPTH,
    eval_metric=EVAL_METRIC,
    auto_class_weights=AUTO_CLASS_WEIGHTS,
    random_seed=SEED,
    verbose=VERBOSE,
)

## Final Training

Train the CatBoost model using the complete metadata dataset.

In [7]:
# Train using the complete metadata dataset.
model.fit(
    X,
    y,
    cat_features=cat_features,
)

0:	total: 133ms	remaining: 2m 12s
100:	total: 6.96s	remaining: 1m 1s
200:	total: 13.2s	remaining: 52.3s
300:	total: 20s	remaining: 46.3s
400:	total: 27.2s	remaining: 40.7s
500:	total: 34.5s	remaining: 34.4s
600:	total: 41.8s	remaining: 27.8s
700:	total: 49.1s	remaining: 21s
800:	total: 56.1s	remaining: 13.9s
900:	total: 1m 1s	remaining: 6.79s
999:	total: 1m 7s	remaining: 0us


CatBoostClassifier(auto_class_weights='Balanced', depth=7, eval_metric='AUC', iterations=1000, learning_rate=0.05, random_seed=42, verbose=100)

## Save Final Model

Export the trained CatBoost model for use during inference and final submission generation.

In [8]:
save_model(
    model,
    f"{MODEL_DIR}/catboost_final.cbm",
)